#***Data Pre-processing***

In [ ]:
DATASET_PATH   = os.path.join("dataset", "dataset_merged") # Corrected path
IMG_SIZE       = (224, 224)          # MobileNetV2 default input size
BATCH_SIZE     = 32                  # Batch size for training
RANDOM_SEED    = 42
SPLIT_DIR      = "split_dataset"    # Directory for train/val/test splits

# Ensure DATASET_PATH exists
if not os.path.exists(DATASET_PATH):
    print(f"Error: The dataset path '{DATASET_PATH}' does not exist.")
    print("Please ensure the ZIP file was extracted correctly in the previous cell.")
else:
    # Get classes
    classes = sorted([d for d in os.listdir(DATASET_PATH)
                     if os.path.isdir(os.path.join(DATASET_PATH, d))])
    class_to_idx = {c: i for i, c in enumerate(classes)}
    NUM_CLASSES = len(classes)
    print(f"Classes detected: {classes}")

    # Count images per class without loading into memory
    class_counts = {}
    for cls in classes:
        cls_dir = os.path.join(DATASET_PATH, cls)
        imgs = [f for f in os.listdir(cls_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        class_counts[cls] = len(imgs)
        print(f"  {cls:<12}: {len(imgs):>5,} images")

    # Calculate class weights for imbalanced dataset
    total_images = sum(class_counts.values())
    class_weights = {
        class_to_idx[cls]: total_images / (NUM_CLASSES * count)
        for cls, count in class_counts.items()
    }

    print("\n--- Class Weights (for handling imbalance) ---")
    for cls, weight in class_weights.items():
        print(f"  {classes[cls]:<12}: {weight:.3f}")

    # ── Create train/val/test splits without loading images ──
    print("\n--- Creating train/val/test splits ---")

    # Create split directories
    for split in ['train', 'val', 'test']:
        for cls in classes:
            os.makedirs(os.path.join(SPLIT_DIR, split, cls), exist_ok=True)

    # Split and copy files
    for cls in classes:
        cls_dir = os.path.join(DATASET_PATH, cls)
        images = [f for f in os.listdir(cls_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

        # Stratified split
        train_imgs, temp_imgs = train_test_split(
            images, test_size=0.30, random_state=RANDOM_SEED, stratify=None
        )
        val_imgs, test_imgs = train_test_split(
            temp_imgs, test_size=0.50, random_state=RANDOM_SEED, stratify=None
        )

        # Copy files (using symlinks would save space, but copy is more reliable)
        for img in train_imgs:
            shutil.copy2(os.path.join(cls_dir, img), os.path.join(SPLIT_DIR, 'train', cls, img))
        for img in val_imgs:
            shutil.copy2(os.path.join(cls_dir, img), os.path.join(SPLIT_DIR, 'val', cls, img))
        for img in test_imgs:
            shutil.copy2(os.path.join(cls_dir, img), os.path.join(SPLIT_DIR, 'test', cls, img))

        print(f"  {cls:<12}: {len(train_imgs):>3} train | {len(val_imgs):>3} val | {len(test_imgs):>3} test")

    # ── Create Data Generators (no pre-loading!) ──
    print("\n--- Creating data generators ---")

    # Training generator with augmentation
    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input_mobilenet,
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.10,
        zoom_range=0.20,
        horizontal_flip=True,
        brightness_range=[0.75, 1.25],
        fill_mode='nearest'
    )

    # Validation and test generators (no augmentation)
    val_test_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input_mobilenet
    )

    # Create generators
    train_generator = train_datagen.flow_from_directory(
        os.path.join(SPLIT_DIR, 'train'),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True,
        seed=RANDOM_SEED
    )

    val_generator = val_test_datagen.flow_from_directory(
        os.path.join(SPLIT_DIR, 'val'),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False,
        seed=RANDOM_SEED
    )

    test_generator = val_test_datagen.flow_from_directory(
        os.path.join(SPLIT_DIR, 'test'),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False,
        seed=RANDOM_SEED
    )

    print(f"\n Generators created successfully!")
    print(f"   Train batches: {len(train_generator)}")
    print(f"   Validation batches: {len(val_generator)}")
    print(f"   Test batches: {len(test_generator)}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"\n Memory efficient: Images are loaded on-the-fly, not pre-loaded!")

Classes detected: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
  cardboard   :   806 images
  glass       : 1,002 images
  metal       :   820 images
  paper       : 1,188 images
  plastic     :   964 images
  trash       :   274 images

--- Class Weights (for handling imbalance) ---
  cardboard   : 1.045
  glass       : 0.841
  metal       : 1.027
  paper       : 0.709
  plastic     : 0.874
  trash       : 3.074

--- Creating train/val/test splits ---
  cardboard   : 564 train | 121 val | 121 test
  glass       : 701 train | 150 val | 151 test
  metal       : 574 train | 123 val | 123 test
  paper       : 831 train | 178 val | 179 test
  plastic     : 674 train | 145 val | 145 test
  trash       : 191 train |  41 val |  42 test

--- Creating data generators ---
Found 3535 images belonging to 6 classes.
Found 758 images belonging to 6 classes.
Found 761 images belonging to 6 classes.

 Generators created successfully!
   Train batches: 111
   Validation batches: 24
   T